# Pilot D: harness-faithful validation on Kaggle (CPU only)
Runs on **CPU**; uses no GPU quota and makes no submission. Part A needs only the competition data and the `metric/gemma-4-developer-agent-wheelhouse` dataset. Part B needs **Internet ON**.

Commit only `pilot_d_summary.json`. Never commit `*_DO_NOT_COMMIT*` files.

In [ ]:
import glob, os, shutil, subprocess, sys, json, time
from pathlib import Path
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
W = Path('/kaggle/working'); W.mkdir(exist_ok=True)
# Same host setup as the official starter notebook (so the subprocess sandbox inherits the same site-packages)
for pth in glob.glob('/usr/local/lib/python*/dist-packages/*cutlass*.pth') + glob.glob('/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    try: os.unlink(pth)
    except OSError: pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    t = tmp_whl / (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
    if not t.exists():
        try: os.symlink(w, t)
        except OSError: shutil.copyfile(w, t)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *sorted(map(str, tmp_whl.glob('*.whl')))], check=True)
print(sys.version)
import swegemma; print('swegemma', swegemma.__file__)

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/__init__.py', 'w').write(''); print('wrote common/__init__.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/kaggle_fetch.py', 'w').write('"""Download individual competition files without a local Kaggle token.\n\nThe session proxy injects the Kaggle credential for *.kaggle.com, so we call the\nsame endpoint the official `kaggle` client uses (competitions.CompetitionApiService/\nDownloadDataFile) and follow the signed-URL redirect. Never commit what this downloads.\n\nUsage: python -I common/kaggle_fetch.py DEST_DIR FILE [FILE ...]\n       python -I common/kaggle_fetch.py --list            (prints every file name + size)\n"""\nimport json\nimport subprocess\nimport sys\nimport urllib.parse\nfrom pathlib import Path\n\nCOMP = "gemma-4-developer-agent"\nAPI = "https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFile"\nLIST = f"https://www.kaggle.com/api/v1/competitions/data/list/{COMP}"\n\n\ndef list_files():\n    tok, out = None, []\n    while True:\n        url = LIST + (f"?pageToken={urllib.parse.quote(tok)}" if tok else "")\n        d = json.loads(subprocess.run(["curl", "-sf", url], capture_output=True, text=True, check=True).stdout)\n        out += d["files"]\n        tok = d.get("nextPageToken")\n        if not tok:\n            return out\n\n\ndef fetch(name: str, dest: Path) -> int:\n    """Two steps: POST for the signed storage URL, then a plain GET (a forced POST on the\n    redirect gets 403 from storage.googleapis.com)."""\n    dest = dest / name\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    body = json.dumps({"competitionName": COMP, "fileName": name})\n    r = subprocess.run(\n        ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code} %{redirect_url}",\n         "-H", "Content-Type: application/json", "--data", body, API],\n        capture_output=True, text=True)\n    code, _, signed = r.stdout.partition(" ")\n    if code != "302" or not signed:\n        return int(code or 0)\n    g = subprocess.run(["curl", "-s", "-o", str(dest), "-w", "%{http_code}", signed],\n                       capture_output=True, text=True)\n    return int(g.stdout or 0)\n\n\nif __name__ == "__main__":\n    if sys.argv[1] == "--list":\n        for f in list_files():\n            print(f["name"], f.get("totalBytes"))\n        sys.exit(0)\n    dest = Path(sys.argv[1])\n    for n in sys.argv[2:]:\n        print(fetch(n, dest), n)\n'); print('wrote common/kaggle_fetch.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/mining.py', 'w').write('"""Mine candidate bug-fix tasks from a local git clone (no GitHub API needed).\n\nEvery filter step is counted so yield can be reported. Test/config classification mirrors\nswegemma.harness.verification._is_protected_test_or_config_path, because that is what the\nscorer strips from agent patches and resets before applying test_patch.\n\nUsage:\n  python -I common/mining.py REPO_DIR --repo owner/name --pkg PKGDIR --since 2026-05-01 \\\n      --out CANDIDATES.jsonl --funnel FUNNEL.json [--max-code-files 3] [--max-code-lines 200]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport subprocess\nfrom pathlib import PurePosixPath\n\nPROTECTED_CONFIG = {"conftest.py", "pytest.ini", "pyproject.toml", "tox.ini", "setup.cfg", ".pytest.ini",\n                    "sitecustomize.py", "usercustomize.py", "_swegemma_stubs.py"}\nFIX_RE = re.compile(r"^(fix|bug|bugfix|hotfix)\\b|^BUG\\b|^fix\\(", re.I)\nISSUE_REF_RE = re.compile(r"\\b(?:fix(?:es|ed)?|close[sd]?|resolve[sd]?)\\s*:?\\s+#(\\d+)", re.I)\nTRAILING_PR_RE = re.compile(r"\\(#(\\d+)\\)\\s*$")\nDOUBLE_REF_RE = re.compile(r"\\(#(\\d+)\\)\\s*\\(#(\\d+)\\)\\s*$")  # sqlglot style: (#issue) (#PR)\nTEST_ISSUE_RE = re.compile(r"(?:\\bgh-|github\\.com/[\\w.-]+/[\\w.-]+/issues/|\\bissue\\s+#?)(\\d{2,6})", re.I)\nAI_RE = re.compile(r"\\[CODEX\\]|Co-authored-by:.*(Claude|Copilot|Codex|Devin|cursor)|Generated with \\[?Claude|"\n                   r"copilot-swe-agent|\\bAI-assisted\\b", re.I)\n\n\ndef is_test_path(p: str) -> bool:\n    name = PurePosixPath(p).name\n    parts = PurePosixPath(p).parts\n    if not p.endswith(".py"):\n        return any(d in ("tests", "test", "testing") for d in parts[:-1])  # fixtures/data under tests/\n    return (name.startswith("test_") or name.endswith("_test.py")\n            or any(d in ("tests", "test", "testing") for d in parts[:-1]))\n\n\ndef is_protected_config(p: str) -> bool:\n    name = PurePosixPath(p).name\n    return name in PROTECTED_CONFIG or name.endswith(".pth")\n\n\ndef git(repo: str, *args: str) -> str:\n    return subprocess.run(["git", "-C", repo, *args], capture_output=True, text=True, check=True).stdout\n\n\ndef to_dev_style(diff: str) -> str:\n    """Rewrite `git diff` output into the dev set\'s style: only \'--- a/X\' / \'+++ b/X\' headers,\n    \'--- a/X\' even for new files, \'+++ b/X\' even for deleted files, no trailing newline."""\n    out, cur_old, cur_new = [], None, None\n    for line in diff.splitlines():\n        if line.startswith("diff --git "):\n            m = re.match(r"diff --git a/(.+) b/(.+)$", line)\n            cur_old, cur_new = m.group(1), m.group(2)\n            continue\n        if line.startswith(("index ", "new file mode", "deleted file mode", "similarity index",\n                            "rename from", "rename to", "old mode", "new mode")):\n            continue\n        if line.startswith("--- "):\n            out.append(f"--- a/{cur_old}")\n            continue\n        if line.startswith("+++ "):\n            out.append(f"+++ b/{cur_new}")\n            continue\n        out.append(line)\n    return "\\n".join(out)\n\n\ndef file_diff(repo: str, parent: str, commit: str, paths: list[str]) -> str:\n    if not paths:\n        return ""\n    return git(repo, "diff", "--no-color", "--no-renames", "-U3", parent, commit, "--", *paths)\n\n\ndef mine(args) -> None:\n    repo = args.repo_dir\n    shas = git(repo, "log", "--first-parent", f"--since={args.since}", "--format=%H", args.branch).split()\n    funnel = {"commits_since": len(shas)}\n    steps = ["single_parent", "fix_message", "has_test_and_code_change", "has_py_test_file", "pkg_python_only",\n             "no_protected_config_in_code", "size_ok"]\n    for s in steps:\n        funnel[s] = 0\n    funnel["with_issue_ref_in_message"] = 0\n    funnel["ai_authored_marker"] = 0\n    out = open(args.out, "w")\n    short = args.repo.split("/")[1].lower()\n    for sha in shas:\n        parents = git(repo, "log", "-1", "--format=%P", sha).split()\n        if len(parents) != 1:\n            continue\n        funnel["single_parent"] += 1\n        subject = git(repo, "log", "-1", "--format=%s", sha).strip()\n        body = git(repo, "log", "-1", "--format=%b", sha)\n        if not FIX_RE.search(subject):\n            continue\n        funnel["fix_message"] += 1\n        parent = parents[0]\n        names = [l.split("\\t") for l in git(repo, "diff", "--no-renames", "--name-status", parent, sha).splitlines()]\n        files = [n[-1] for n in names]\n        test_files = [f for f in files if is_test_path(f)]\n        code_files = [f for f in files if not is_test_path(f)]\n        code_py = [f for f in code_files if f.endswith(".py")]\n        if not test_files or not code_py:\n            continue\n        funnel["has_test_and_code_change"] += 1\n        # The harness runs pytest on the .py files of test_patch; fixture-only test patches\n        # make it pass a non-.py path to pytest ("no match"), so require a .py test file.\n        if not any(f.endswith(".py") and PurePosixPath(f).name != "conftest.py" for f in test_files):\n            continue\n        funnel["has_py_test_file"] += 1\n        # Code change must be Python inside the package; allow docs/changelog alongside (dropped from gold).\n        pkg_files = [f for f in code_files if f.startswith(args.pkg + "/")]\n        if any(not f.endswith(".py") for f in pkg_files) or not all(f in pkg_files for f in code_py):\n            continue\n        funnel["pkg_python_only"] += 1\n        if any(is_protected_config(f) for f in code_files):\n            continue\n        funnel["no_protected_config_in_code"] += 1\n        code_diff = file_diff(repo, parent, sha, pkg_files)\n        added = sum(1 for l in code_diff.splitlines() if l.startswith(("+", "-")) and not l.startswith(("+++", "---")))\n        if len(pkg_files) > args.max_code_files or added > args.max_code_lines:\n            continue\n        funnel["size_ok"] += 1\n        m2 = DOUBLE_REF_RE.search(subject)\n        issue_refs = sorted({int(x) for x in ISSUE_REF_RE.findall(subject + "\\n" + body)}\n                            | ({int(m2.group(1))} if m2 else set()))\n        pr = int(m2.group(2)) if m2 else (int(TRAILING_PR_RE.search(subject).group(1)) if TRAILING_PR_RE.search(subject) else None)\n        issue_refs = [i for i in issue_refs if i != pr]\n        test_diff = file_diff(repo, parent, sha, test_files)\n        added_test_text = "\\n".join(l for l in test_diff.splitlines() if l.startswith("+"))\n        issue_refs_in_tests = sorted({int(x) for x in TEST_ISSUE_RE.findall(added_test_text)} - {pr})\n        funnel["with_issue_ref_in_tests"] = funnel.get("with_issue_ref_in_tests", 0) + bool(issue_refs_in_tests)\n        funnel["with_any_issue_ref"] = funnel.get("with_any_issue_ref", 0) + bool(issue_refs or issue_refs_in_tests)\n        funnel["with_issue_ref_in_message"] += bool(issue_refs)\n        ai = bool(AI_RE.search(subject + "\\n" + body))\n        funnel["ai_authored_marker"] += ai\n        rec = {\n            "instance_id": f"{short}_{pr or sha[:10]}",\n            "repo": args.repo,\n            "base_commit": parent,\n            "fix_commit": sha,\n            "pr_number": pr,\n            "issue_refs": issue_refs,\n            "issue_refs_in_tests": issue_refs_in_tests,\n            "subject": subject,\n            "committed_at": git(repo, "log", "-1", "--format=%cI", sha).strip(),\n            "authored_at": git(repo, "log", "-1", "--format=%aI", sha).strip(),\n            "code_files": pkg_files,\n            "test_files": test_files,\n            "code_lines_changed": added,\n            "ai_authored_marker": ai,\n            "patch": to_dev_style(code_diff),\n            "test_patch": to_dev_style(test_diff),\n        }\n        out.write(json.dumps(rec) + "\\n")\n    json.dump(funnel, open(args.funnel, "w"), indent=1)\n    print(json.dumps(funnel))\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("repo_dir")\n    p.add_argument("--repo", required=True)\n    p.add_argument("--pkg", required=True)\n    p.add_argument("--since", default="2026-05-01")\n    p.add_argument("--branch", default="HEAD")\n    p.add_argument("--out", required=True)\n    p.add_argument("--funnel", required=True)\n    p.add_argument("--max-code-files", type=int, default=3)\n    p.add_argument("--max-code-lines", type=int, default=200)\n    mine(p.parse_args())\n'); print('wrote common/mining.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/snapshot.py', 'w').write('"""Build a task snapshot .tgz in the same layout as the competition\'s snapshots/<instance_id>.tgz.\n\nObserved dev layout (schema only): archive rooted at \'./\', a git repo whose history is\n`git fast-export` of the source up to base_commit piped into `git fast-import` (so commit hashes\ndiffer from upstream; tasks.jsonl keeps the upstream base_commit), branches `export_ref` and\n`main` (checked out) at the base, .git/config with user agent@eval / Agent, clean working tree.\nHistory depth is whatever the source clone has (dev snapshots range from 4 to 6k commits).\n\nUsage: python -I common/snapshot.py REPO_DIR BASE_COMMIT OUT.tgz\n"""\nfrom __future__ import annotations\n\nimport subprocess\nimport sys\nimport tempfile\nfrom pathlib import Path\n\n\ndef run(*cmd: str, cwd: str | None = None, stdin=None, stdout=None) -> None:\n    subprocess.run(cmd, cwd=cwd, check=True, stdin=stdin, stdout=stdout, stderr=subprocess.PIPE)\n\n\ndef build_snapshot(repo_dir: str, base_commit: str, out_tgz: str) -> dict:\n    out = Path(out_tgz)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory() as td:\n        dst = Path(td) / "ws"\n        dst.mkdir()\n        run("git", "init", "-q", "-b", "main", str(dst))\n        run("git", "-C", repo_dir, "branch", "-f", "export_ref", base_commit)\n        exp = subprocess.Popen(["git", "-C", repo_dir, "fast-export", "--signed-tags=strip",\n                                "--tag-of-filtered-object=drop", "--reencode=yes", "refs/heads/export_ref"],\n                               stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)\n        run("git", "-C", str(dst), "fast-import", "--quiet", stdin=exp.stdout)\n        if exp.wait() != 0:\n            raise RuntimeError("git fast-export failed")\n        run("git", "-C", str(dst), "checkout", "-q", "-B", "main", "export_ref")\n        run("git", "-C", str(dst), "config", "user.email", "agent@eval")\n        run("git", "-C", str(dst), "config", "user.name", "Agent")\n        run("git", "-C", str(dst), "gc", "-q", "--prune=now")\n        # Same top-level shape as dev archives: entries prefixed with \'./\'\n        run("tar", "-C", str(dst), "-czf", str(out), ".")\n        n_commits = int(subprocess.run(["git", "-C", str(dst), "rev-list", "--count", "HEAD"],\n                                       capture_output=True, text=True).stdout.strip() or 0)\n    return {"snapshot_bytes": out.stat().st_size, "history_commits": n_commits}\n\n\nif __name__ == "__main__":\n    print(build_snapshot(sys.argv[1], sys.argv[2], sys.argv[3]))\n'); print('wrote common/snapshot.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/validate.py', 'w').write('"""Validate tasks with the competition harness\'s own Phase-2 code (swegemma.verify_task).\n\nA task is VALID iff, in the chosen sandbox backend:\n  baseline: verify_task(agent_patch="")          -> resolved == False   (tests fail without the fix)\n  gold:     verify_task(agent_patch=task.patch)  -> resolved == True    (exit 0 + JUnit checks pass)\nThis is the host\'s own criterion (swegemma.harness.sample_verification: base_exit != 0 and\npatch_exit == 0), but run through verify_task so protected-file stripping, test-file reset,\npytest.ini/conftest rewriting and JUnit validation are exactly what the scorer does.\n\nRun with the interpreter that has swegemma installed. In subprocess mode that interpreter\'s\nsite-packages ARE the sandbox\'s environment (SubprocessManager uses system_site_packages=True),\nso it should mirror the Kaggle notebook host as closely as possible.\n\nUsage:\n  python common/validate.py --tasks T.jsonl --snapshots DIR --backend subprocess \\\n      --out results.jsonl [--wheels DIR] [--ids a b c] [--logs DIR]\nOutput: one JSON line per task with verdict fields only (no patches, no test output).\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport asyncio\nimport json\nimport logging\nimport sys\nimport tempfile\nimport time\nfrom pathlib import Path\n\n\ndef _config(args, results_dir: Path):\n    from adk_submission import ModelRegistry\n    from swegemma.config import EvalConfig\n\n    return EvalConfig(\n        tasks_path=Path(args.tasks),\n        snapshots_dir=Path(args.snapshots),\n        results_dir=results_dir,\n        submission_dir=results_dir,\n        models=ModelRegistry(),\n        sandbox=args.backend,\n        image=args.image,\n        wheels_dir=Path(args.wheels) if args.wheels else None,\n        timeout_seconds=args.timeout,\n        verification_timeout_seconds=args.timeout,\n    )\n\n\ndef _sandbox(config):\n    if config.sandbox == "docker":\n        from swegemma.sandbox import ContainerConfig, ContainerManager\n\n        return ContainerManager(ContainerConfig(image=config.image, timeout_seconds=config.verification_timeout_seconds))\n    from swegemma.sandbox.subprocess import SubprocessManager\n\n    return SubprocessManager(timeout_seconds=config.verification_timeout_seconds)\n\n\ndef _first_error_line(text: str) -> str:\n    """A short, data-free failure category (exception class / pytest summary keyword)."""\n    import re\n\n    for pat in (r"(ModuleNotFoundError|ImportError|AttributeError|TypeError|NameError|SyntaxError|"\n                r"AssertionError|FileNotFoundError|fixture \'[^\']+\' not found|ERROR collecting)",\n                r"(\\d+ failed|\\d+ error)"):\n        m = re.search(pat, text or "")\n        if m:\n            return m.group(1)[:80]\n    return ""\n\n\nasync def _one(task, args, logs: Path | None):\n    from swegemma.harness.verification import verify_task\n\n    snap = Path(args.snapshots) / f"{task.instance_id}.tgz"\n    if not snap.exists():\n        snap = Path(args.snapshots) / f"{task.instance_id}.tar.gz"\n    rec = {"instance_id": task.instance_id, "repo": task.repo, "backend": args.backend}\n    with tempfile.TemporaryDirectory() as td:\n        config = _config(args, Path(td))\n        sb = _sandbox(config)\n        for kind, patch in (("base", ""), ("gold", task.patch)):\n            t0 = time.perf_counter()\n            try:\n                r = await verify_task(sb, config, task, snap, agent_patch=patch, start_time=t0)\n                rec[f"{kind}_resolved"] = bool(r.resolved)\n                rec[f"{kind}_exit"] = r.test_exit_code\n                rec[f"{kind}_error"] = (r.error or "")[:160]\n                rec[f"{kind}_symptom"] = _first_error_line(r.test_output)\n                if logs:\n                    (logs / f"{task.instance_id}.{args.backend}.{kind}.log").write_text(\n                        f"error: {r.error}\\n\\n{r.test_output}")\n            except Exception as e:  # harness/setup failure, not a test verdict\n                rec[f"{kind}_resolved"] = None\n                rec[f"{kind}_error"] = f"{type(e).__name__}: {e}"[:160]\n            rec[f"{kind}_seconds"] = round(time.perf_counter() - t0, 1)\n    rec["valid"] = rec.get("base_resolved") is False and rec.get("gold_resolved") is True\n    return rec\n\n\nasync def main_async(args):\n    from swegemma.models import load_tasks\n\n    tasks = load_tasks(Path(args.tasks))\n    if args.ids:\n        tasks = [t for t in tasks if t.instance_id in set(args.ids)]\n    logs = Path(args.logs) if args.logs else None\n    if logs:\n        logs.mkdir(parents=True, exist_ok=True)\n    out = open(args.out, "a")\n    sem = asyncio.Semaphore(args.concurrency)\n\n    async def run(t):\n        async with sem:\n            rec = await _one(t, args, logs)\n            out.write(json.dumps(rec) + "\\n")\n            out.flush()\n            print(json.dumps({k: rec.get(k) for k in ("instance_id", "valid", "base_resolved", "gold_resolved",\n                                                      "gold_symptom", "gold_error", "gold_seconds")}), flush=True)\n\n    await asyncio.gather(*(run(t) for t in tasks))\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("--tasks", required=True)\n    p.add_argument("--snapshots", required=True)\n    p.add_argument("--backend", choices=["subprocess", "docker"], default="subprocess")\n    p.add_argument("--image", default="swebench-sandbox:latest")\n    p.add_argument("--wheels")\n    p.add_argument("--ids", nargs="*")\n    p.add_argument("--out", required=True)\n    p.add_argument("--logs")\n    p.add_argument("--timeout", type=int, default=300)\n    p.add_argument("--concurrency", type=int, default=2)\n    a = p.parse_args()\n    logging.basicConfig(level=logging.WARNING)\n    asyncio.run(main_async(a))\n'); print('wrote common/validate.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/testmap.py', 'w').write('"""Compute FAIL_TO_PASS / PASS_TO_PASS with the harness\'s own sandbox setup.\n\nFor each task: fresh sandbox -> extract snapshot -> harness test config -> baseline commit ->\napply test_patch -> pytest (JUnit) [state "base"]; then apply the gold patch -> pytest [state "gold"].\nF2P = tests not passing at base and passing at gold; P2P = passing in both.\nNode ids use the SWE-bench convention `path::Class::test[param]`.\n\nUsage: python common/testmap.py --tasks T.jsonl --snapshots DIR --out TESTMAP.jsonl [--ids ...]\n(run with the interpreter that has swegemma installed; subprocess backend)\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport tempfile\nimport xml.etree.ElementTree as ET\nfrom pathlib import Path\n\nfrom swegemma.harness.container_setup import (apply_patch_in_container, extract_snapshot, setup_baseline_commit,\n                                              setup_git_exclude, setup_workspace_test_config)\nfrom swegemma.models import load_tasks\nfrom swegemma.sandbox.subprocess import SubprocessManager\n\n\n# Per-test timeout (pytest-timeout) used ONLY for labelling: a hang bug otherwise kills the whole\n# base run, no JUnit is written, and every test of the file would be mislabelled FAIL_TO_PASS.\n# Validity verdicts still come from verify_task, which runs with the harness\'s own timeout=0.\nPER_TEST_TIMEOUT = 60\n\n\ndef _outcomes(sb, sid, targets: list[str]) -> dict[str, str]:\n    xml = "/tmp/_testmap_junit.xml"\n    cmd = (f"cd /workspace && rm -f {xml} && PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 python3 -s -m pytest "\n           f"{\' \'.join(targets)} --junitxml={xml} -o junit_family=xunit1 -p no:anyio -o timeout={PER_TEST_TIMEOUT} "\n           f"-o python_classes=\'Test* *Test\' -q >/dev/null 2>&1; cat {xml}")\n    out = sb.exec(sid, cmd).stdout or ""\n    res: dict[str, str] = {}\n    try:\n        root = ET.fromstring(out[out.find("<"):])\n    except ET.ParseError:\n        return res\n    for tc in root.iter("testcase"):\n        f, cls, name = tc.get("file", ""), tc.get("classname", ""), tc.get("name", "")\n        mod = f[:-3].replace("/", ".") if f.endswith(".py") else ""\n        inner = cls[len(mod) + 1:] if mod and cls.startswith(mod + ".") else ""\n        node = "::".join(x for x in (f, inner.replace(".", "::"), name) if x)\n        state = ("failed" if tc.find("failure") is not None else "error" if tc.find("error") is not None\n                 else "skipped" if tc.find("skipped") is not None else "passed")\n        res[node] = state\n    return res\n\n\ndef _apply(sb, sid, patch: str) -> int:\n    with tempfile.NamedTemporaryFile("w", suffix=".patch", delete=False) as fh:\n        fh.write(patch if patch.endswith("\\n") else patch + "\\n")\n    sb.copy_to(sid, Path(fh.name), "/tmp/")\n    code, _, _ = apply_patch_in_container(sb, sid, f"/tmp/{Path(fh.name).name}")\n    Path(fh.name).unlink(missing_ok=True)\n    return code\n\n\ndef testmap(task, snap: Path) -> dict:\n    sb = SubprocessManager(timeout_seconds=600)\n    sid = sb.start()\n    try:\n        extract_snapshot(sb, sid, snap)\n        setup_git_exclude(sb, sid)\n        setup_workspace_test_config(sb, sid, repo=task.repo)\n        setup_baseline_commit(sb, sid, "eval_baseline")\n        targets = [p for p in task.test_files if p.endswith(".py") and not p.endswith("conftest.py")]\n        if _apply(sb, sid, task.test_patch) != 0:\n            return {"instance_id": task.instance_id, "error": "test_patch did not apply"}\n        base = _outcomes(sb, sid, targets)\n        if _apply(sb, sid, task.patch) != 0:\n            return {"instance_id": task.instance_id, "error": "gold patch did not apply"}\n        gold = _outcomes(sb, sid, targets)\n    finally:\n        sb.stop(sid)\n    f2p = sorted(n for n, s in gold.items() if s == "passed" and base.get(n) != "passed")\n    p2p = sorted(n for n, s in gold.items() if s == "passed" and base.get(n) == "passed")\n    broken = sorted(n for n, s in gold.items() if s in ("failed", "error"))\n    skipped = sorted(n for n, s in gold.items() if s == "skipped")\n    return {"instance_id": task.instance_id, "FAIL_TO_PASS": f2p, "PASS_TO_PASS": p2p,\n            "gold_not_passing": broken, "gold_skipped": skipped, "n_tests": len(gold)}\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("--tasks", required=True)\n    p.add_argument("--snapshots", required=True)\n    p.add_argument("--out", required=True)\n    p.add_argument("--ids", nargs="*")\n    a = p.parse_args()\n    with open(a.out, "w") as fh:\n        for t in load_tasks(Path(a.tasks)):\n            if a.ids and t.instance_id not in a.ids:\n                continue\n            rec = testmap(t, Path(a.snapshots) / f"{t.instance_id}.tgz")\n            fh.write(json.dumps(rec) + "\\n")\n            print(t.instance_id, {k: (len(v) if isinstance(v, list) else v) for k, v in rec.items() if k != "instance_id"},\n                  flush=True)\n'); print('wrote common/testmap.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/annotate.py', 'w').write('"""Intervention-ready annotations for swegemma-format tasks (held-out or dev).\n\nPer task:\n  edit_hunks:   each code-patch hunk -> file, old line span, enclosing symbol as a graph node id\n                (released id rule, common/graph_build.released_ids_for_file) or "<module>" if at module level\n  gold_nodes:   unique node ids edited by the gold patch (localization ground truth, graph id space)\n  gold_nodes_missing_from_graph_rule: edited symbols the released-style graph cannot contain\n                (module-level code, async defs)\n  added_symbols: defs/classes that only exist after the patch (new functions)\n  repro_test:   (needs FAIL_TO_PASS) a minimal runnable pytest file: test-file imports and module-level\n                statements + only the F2P tests (TestCase classes keep setUp/fixtures/helpers), with\n                --verify: run in the harness sandbox, must FAIL before and PASS after the gold patch.\n\nUsage (interpreter with swegemma + codegraph-loc installed):\n  python common/annotate.py --tasks T.jsonl --snapshots DIR --out ANN.jsonl [--testmap TM.jsonl] [--verify]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport io\nimport json\nimport re\nimport sys\nimport tarfile\nimport tempfile\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom common.graph_build import released_ids_for_file  # noqa: E402\n\nHUNK_RE = re.compile(r"^@@ -(\\d+)(?:,(\\d+))? \\+(\\d+)(?:,(\\d+))? @@")\n\n\ndef parse_hunks(patch: str):\n    """Yield (file, old_start, old_len, new_start, new_len, changed_old_lines, changed_new_lines)."""\n    cur, h = None, None\n    for line in patch.splitlines():\n        if line.startswith("+++ b/"):\n            cur = line[6:]\n            continue\n        if line.startswith("--- "):\n            continue\n        m = HUNK_RE.match(line)\n        if m:\n            if h:\n                yield h\n            os_, ol, ns, nl = int(m[1]), int(m[2] or 1), int(m[3]), int(m[4] or 1)\n            h = {"file": cur, "old_start": os_, "old_len": ol, "new_start": ns, "new_len": nl,\n                 "old_changed": [], "new_changed": [], "_o": os_, "_n": ns}\n            continue\n        if h is None:\n            continue\n        if line.startswith("-"):\n            h["old_changed"].append(h["_o"]); h["_o"] += 1\n        elif line.startswith("+"):\n            h["new_changed"].append(h["_n"]); h["_n"] += 1\n        else:\n            h["_o"] += 1; h["_n"] += 1\n    if h:\n        yield h\n\n\ndef _defs_with_spans(src: str):\n    try:\n        tree = ast.parse(src)\n    except SyntaxError:\n        return []\n    out = []\n    for n in ast.walk(tree):\n        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):\n            start = min([n.lineno] + [d.lineno for d in n.decorator_list])\n            out.append((start, n.end_lineno, n.lineno, isinstance(n, ast.AsyncFunctionDef)))\n    return out\n\n\ndef enclosing(defs, line):\n    best = None\n    for s, e, defline, is_async in defs:\n        if s <= line <= e and (best is None or s >= best[0]):\n            best = (s, e, defline, is_async)\n    return best\n\n\ndef module_name(root: Path, rel: str) -> str:\n    from cgl.graph import module_name_for\n    return module_name_for(root / rel, root, scheme="official")[0]\n\n\ndef locate(root: Path, task: dict) -> dict:\n    hunks, gold, missing = [], set(), set()\n    new_files = {}\n    for h in parse_hunks(task["patch"]):\n        rel = h["file"]\n        if not rel.endswith(".py"):\n            continue\n        f = root / rel\n        mod = module_name(root, rel)\n        src = f.read_text(encoding="utf-8", errors="replace") if f.exists() else ""\n        defs, ids = _defs_with_spans(src), released_ids_for_file(f, mod) if f.exists() else {}\n        lines = h["old_changed"] or [max(1, h["old_start"] + (h["old_len"] - 1) // 2)]  # pure insertion: anchor\n        syms = set()\n        for ln in lines:\n            enc = enclosing(defs, ln)\n            if enc is None:\n                syms.add(f"{mod}.<module>"); missing.add(f"{mod}.<module>")\n            elif enc[2] in ids:\n                syms.add(ids[enc[2]])\n            else:  # async def or other def the released graphs do not contain\n                tag = f"{mod}.<async@{enc[2]}>" if enc[3] else f"{mod}.<def@{enc[2]}>"\n                syms.add(tag); missing.add(tag)\n        gold |= {s for s in syms if "<" not in s}\n        hunks.append({"file": rel, "old_start": h["old_start"], "old_len": h["old_len"], "symbols": sorted(syms)})\n        new_files.setdefault(rel, []).extend(h["new_changed"])\n    return {"edit_hunks": hunks, "gold_nodes": sorted(gold),\n            "gold_nodes_missing_from_graph_rule": sorted(missing), "_new_lines": new_files}\n\n\ndef added_symbols(new_root: Path, new_lines: dict, old_ids: set[str]) -> list[str]:\n    out = set()\n    for rel, lines in new_lines.items():\n        f = new_root / rel\n        if not f.exists():\n            continue\n        ids = released_ids_for_file(f, module_name(new_root, rel))\n        defs = _defs_with_spans(f.read_text(encoding="utf-8", errors="replace"))\n        for ln in lines:\n            enc = enclosing(defs, ln)\n            if enc and enc[2] in ids and ids[enc[2]] not in old_ids:\n                out.add(ids[enc[2]])\n    return sorted(out)\n\n\n# ---------------- reproduction tests ----------------\n\ndef build_repro(test_src: str, f2p_nodes: list[str], test_path: str) -> str | None:\n    """Keep imports + module-level non-test statements + only the selected tests."""\n    tree = ast.parse(test_src)\n    want_funcs, want_methods = set(), {}\n    for node in f2p_nodes:\n        parts = node.split("::")\n        if parts[0] != test_path:\n            continue\n        name = parts[-1].split("[", 1)[0]\n        if len(parts) == 2:\n            want_funcs.add(name)\n        else:\n            want_methods.setdefault(parts[1], set()).add(name)\n    if not want_funcs and not want_methods:\n        return None\n    body = []\n    for st in tree.body:\n        if isinstance(st, (ast.FunctionDef, ast.AsyncFunctionDef)):\n            if st.name.startswith("test"):\n                if st.name in want_funcs:\n                    body.append(st)\n            else:\n                body.append(st)  # helpers and fixtures\n        elif isinstance(st, ast.ClassDef):\n            if st.name in want_methods:\n                keep = [b for b in st.body if not (isinstance(b, (ast.FunctionDef, ast.AsyncFunctionDef))\n                                                   and b.name.startswith("test") and b.name not in want_methods[st.name])]\n                st.body = keep or [ast.Pass()]\n                body.append(st)\n            elif not st.name.startswith("Test"):\n                body.append(st)  # helper / base classes\n        else:\n            body.append(st)\n    tree.body = body\n    return ast.unparse(tree) + "\\n"\n\n\ndef _sandbox_run_repro(snap: Path, repo: str, repro: str, gold_patch: str, test_patch: str):\n    from swegemma.harness.container_setup import (apply_patch_in_container, extract_snapshot, setup_baseline_commit,\n                                                  setup_git_exclude, setup_workspace_test_config)\n    from swegemma.sandbox.subprocess import SubprocessManager\n\n    def apply(sb, sid, patch):\n        with tempfile.NamedTemporaryFile("w", suffix=".patch", delete=False) as fh:\n            fh.write(patch if patch.endswith("\\n") else patch + "\\n")\n        sb.copy_to(sid, Path(fh.name), "/tmp/")\n        rc = apply_patch_in_container(sb, sid, f"/tmp/{Path(fh.name).name}")[0]\n        Path(fh.name).unlink(missing_ok=True)\n        return rc\n\n    sb = SubprocessManager(timeout_seconds=600)\n    sid = sb.start()\n    try:\n        extract_snapshot(sb, sid, snap)\n        setup_git_exclude(sb, sid)\n        setup_workspace_test_config(sb, sid, repo=repo)\n        setup_baseline_commit(sb, sid, "eval_baseline")\n        # test_patch may add fixtures/data the repro needs (it never adds library code)\n        apply(sb, sid, test_patch)\n        sb.write_file("/tmp/test_repro_injected.py", repro)\n        cmd = ("cd /workspace && PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 timeout 240 python3 -s -m pytest "\n               "/tmp/test_repro_injected.py -p no:anyio -p no:cacheprovider -o timeout=60 -q >/dev/null 2>&1; echo $?")\n        before = (sb.exec(sid, cmd).stdout or "").strip().splitlines()[-1:]\n        apply(sb, sid, gold_patch)\n        after = (sb.exec(sid, cmd).stdout or "").strip().splitlines()[-1:]\n    finally:\n        sb.stop(sid)\n    return (before[0] if before else "?"), (after[0] if after else "?")\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--tasks", required=True)\n    p.add_argument("--snapshots", required=True)\n    p.add_argument("--out", required=True)\n    p.add_argument("--testmap")\n    p.add_argument("--verify", action="store_true")\n    p.add_argument("--ids", nargs="*")\n    a = p.parse_args()\n    tm = {r["instance_id"]: r for r in map(json.loads, open(a.testmap))} if a.testmap else {}\n    with open(a.out, "w") as out:\n        for line in open(a.tasks):\n            t = json.loads(line)\n            if a.ids and t["instance_id"] not in a.ids:\n                continue\n            snap = Path(a.snapshots) / f"{t[\'instance_id\']}.tgz"\n            with tempfile.TemporaryDirectory() as td:\n                root = Path(td) / "ws"\n                with tarfile.open(snap) as tf:\n                    tf.extractall(root, filter="data")\n                rec = {"instance_id": t["instance_id"], **locate(root, t)}\n                # apply gold + test patch in a scratch copy to find added symbols and the post-test file\n                import subprocess\n                for patch in (t["patch"], t["test_patch"]):\n                    pf = Path(td) / "p.patch"\n                    pf.write_text(patch + ("\\n" if not patch.endswith("\\n") else ""))\n                    subprocess.run(["git", "-c", "safe.directory=*", "-C", str(root), "apply", "--unsafe-paths",\n                                    str(pf)], capture_output=True)\n                rec["added_symbols"] = added_symbols(root, rec.pop("_new_lines"), set(rec["gold_nodes"]))\n                f2p = tm.get(t["instance_id"], {}).get("FAIL_TO_PASS", [])\n                if f2p:\n                    test_path = f2p[0].split("::")[0]\n                    src = (root / test_path).read_text(encoding="utf-8", errors="replace")\n                    repro = build_repro(src, f2p, test_path)\n                    rec["repro_test"] = {"source_file": test_path, "n_f2p": len(f2p), "code": repro}\n                    if repro and a.verify:\n                        rc_before, rc_after = _sandbox_run_repro(snap, t["repo"], repro, t["patch"], t["test_patch"])\n                        rec["repro_test"].update(exit_before_gold=rc_before, exit_after_gold=rc_after,\n                                                 verified=(rc_before not in ("0", "?") and rc_after == "0"))\n            out.write(json.dumps(rec) + "\\n")\n            print(t["instance_id"], len(rec["gold_nodes"]), "gold nodes;",\n                  "missing", len(rec["gold_nodes_missing_from_graph_rule"]),\n                  "| repro verified:", rec.get("repro_test", {}).get("verified"), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote common/annotate.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/graph_build.py', 'w').write('"""Code graphs in the competition\'s released node-link schema, built with codegraph-loc.\n\nGenerator: codegraph-loc (Apache-2.0, https://github.com/danielmantiglia/codegraph-loc,\ncommit 1e8e0ab, D. Mantiglia 2026), used as a library: `cgl.graph.build_graph` +\n`cgl.graph.official_view` (released id convention). This module adds a "released view"\nreverse-engineered from the released graphs (schema notes in docs/FORMAT.md):\n  * nodes: classes, functions, methods only (no module nodes); async defs dropped unless --fixed\n  * node text: functions from the `def` keyword (original indentation kept); classes rendered as\n    a skeleton: header, cleaned docstring (or `None`), then method signatures with `...` bodies\n  * edges: type "calls", key 0, and SYMMETRIC: every containment pair and every resolved call is\n    stored in both directions (in the released graphs 100% of edges have their reverse)\n  * graph attribute repo_name = "<repo_short>_<base_commit>"; file name graphs/<repo_short>_<base_commit>.json\n\nUsage:\n  python common/graph_build.py build REPO_ROOT REPO_SHORT BASE_COMMIT OUT_DIR [--fixed]\n  python common/graph_build.py fidelity OURS.json RELEASED.json\n"""\nfrom __future__ import annotations\n\nimport ast\nimport json\nimport sys\nfrom pathlib import Path\n\n\ndef _func_sig(n) -> str:\n    s = f"def {n.name}({ast.unparse(n.args)})"\n    if n.returns is not None:\n        s += f" -> {ast.unparse(n.returns)}"\n    return s + ":"\n\n\ndef _class_text(node: ast.ClassDef) -> str:\n    bases = [ast.unparse(b) for b in node.bases] + [ast.unparse(k) for k in node.keywords]\n    head = f"class {node.name}" + (f"({\', \'.join(bases)})" if bases else "") + ":"\n    doc = ast.get_docstring(node)\n    parts = [head, f\'    """{doc}"""\' if doc is not None else "    None", ""]\n    for b in node.body:\n        if isinstance(b, (ast.FunctionDef, ast.AsyncFunctionDef)):\n            for d in b.decorator_list:\n                parts.append(f"    @{ast.unparse(d)}")\n            parts.append(f"    {_func_sig(b)}\\n        ...\\n")\n    return "\\n".join(parts).rstrip()\n\n\ndef _texts_for_file(path: Path) -> dict[int, str]:\n    """Map definition line (the `def`/`class` keyword line) -> released-style text."""\n    src = path.read_text(encoding="utf-8", errors="replace")\n    try:\n        tree = ast.parse(src)\n    except SyntaxError:\n        return {}\n    out = {}\n    for n in ast.walk(tree):\n        if isinstance(n, ast.ClassDef):\n            out[n.lineno] = _class_text(n)\n        elif isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)):\n            seg = ast.get_source_segment(src, n) or ""\n            out[n.lineno] = seg[seg.find("def ") if not seg.startswith("def ") else 0:] if "def " in seg else seg\n    return out\n\n\nRELEASED_CALL_RESOLUTIONS = {"module", "import", "local"}  # no self./typed/unique-name/super in released graphs\n\n\ndef released_ids_for_file(path: Path, module: str, fixed: bool = False) -> dict[int, str]:\n    """Exact released id rule (reproduces 100% of node ids on the released requests/httpx graphs).\n\n    The released generator keeps ONE "current class" variable, set on entering a class body and\n    reset to None (not restored) on leaving it. So: classes -> <module>.<Class>; any def while a\n    class is current -> <module>.<Class>.<name> (closures inside methods included); any def after a\n    nested class body has closed inside an outer class -> <module>.<name> (the documented\n    "methods labelled as module-level" bug). Async defs are skipped (released graphs have none).\n    With fixed=True a proper class stack is used and async defs are kept.\n    Returns {line of def/class keyword (and of each decorator): id}."""\n    try:\n        tree = ast.parse(path.read_text(encoding="utf-8", errors="replace"))\n    except SyntaxError:\n        return {}\n    out: dict[int, str] = {}\n    state = {"cls": None}\n    func_types = (ast.FunctionDef, ast.AsyncFunctionDef) if fixed else (ast.FunctionDef,)\n\n    def mark(node, nid):\n        out[node.lineno] = nid\n        for d in node.decorator_list:\n            out.setdefault(d.lineno, nid)\n\n    def visit(node):\n        for ch in ast.iter_child_nodes(node):\n            if isinstance(ch, ast.ClassDef):\n                mark(ch, f"{module}.{ch.name}")\n                prev = state["cls"]\n                state["cls"] = ch.name\n                visit(ch)\n                state["cls"] = prev if fixed else None\n            elif isinstance(ch, func_types):\n                mark(ch, f"{module}.{state[\'cls\']}.{ch.name}" if state["cls"] else f"{module}.{ch.name}")\n                visit(ch)\n            else:\n                visit(ch)\n\n    visit(tree)\n    return out\n\n\ndef build(root: str, repo_short: str, base_commit: str, out_dir: str, fixed: bool = False) -> Path:\n    """Node ids/texts from our AST rules; call resolution from codegraph-loc, re-keyed by (file, line)."""\n    from cgl.graph import build_graph\n\n    G0 = build_graph(root, module_scheme="official")  # package-relative names inside packages, else repo path\n    rootp = Path(root)\n    ids_cache: dict[str, dict[int, str]] = {}\n    texts_cache: dict[str, dict[int, str]] = {}\n    mapping, keep = {}, {}\n    for n, a in G0.nodes(data=True):\n        if a.get("kind") not in ("class", "function", "method") or not a.get("file"):\n            continue\n        f = a["file"]\n        if f not in ids_cache:\n            ids_cache[f] = released_ids_for_file(rootp / f, a.get("module"), fixed)\n            texts_cache[f] = _texts_for_file(rootp / f)\n        sl = a.get("start_line") or 0\n        rid = ids_cache[f].get(sl)\n        if rid is None:\n            continue  # e.g. async def in faithful mode\n        mapping[n] = rid\n        if rid not in keep:  # released graphs collapse same-id defs into one node (first wins)\n            text = next((texts_cache[f][ln] for ln in range(sl, sl + 50) if ln in texts_cache[f]), a.get("text", ""))\n            keep[rid] = {"id": rid, "name": rid, "text": text}\n    kind = {n: a.get("kind") for n, a in G0.nodes(data=True)}\n    pairs = set()\n    for u, v, a in G0.edges(data=True):\n        if u not in mapping or v not in mapping:\n            continue\n        t = a.get("type")\n        ok = (t == "calls" and (fixed or a.get("resolution") in RELEASED_CALL_RESOLUTIONS)) or \\\n             (t == "contains" and (fixed or (kind.get(u) == "class" and kind.get(v) == "method")))\n        mu, mv = mapping[u], mapping[v]\n        if ok and mu != mv:\n            pairs.add((mu, mv))\n            pairs.add((mv, mu))\n    data = {"directed": True, "multigraph": True, "graph": {"repo_name": f"{repo_short}_{base_commit}"},\n            "nodes": list(keep.values()),\n            "edges": [{"type": "calls", "source": u, "target": v, "key": 0} for u, v in sorted(pairs)]}\n    out = Path(out_dir) / f"{repo_short}_{base_commit}.json"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(data))\n    return out\n\n\ndef fidelity(ours_path: str, released_path: str) -> dict:\n    o, r = json.load(open(ours_path)), json.load(open(released_path))\n    on, rn = {n["id"]: n for n in o["nodes"]}, {n["id"]: n for n in r["nodes"]}\n    oe = {frozenset((e["source"], e["target"])) for e in o["edges"]}\n    re_ = {frozenset((e["source"], e["target"])) for e in r["edges"]}\n    common = set(on) & set(rn)\n    re_in = {e for e in re_ if e <= set(on)}  # released edges whose endpoints we also have\n    return {\n        "released_nodes": len(rn), "our_nodes": len(on),\n        "node_recall": round(len(common) / max(1, len(rn)), 4),\n        "node_precision": round(len(common) / max(1, len(on)), 4),\n        "text_exact_on_common": round(sum(on[i]["text"] == rn[i]["text"] for i in common) / max(1, len(common)), 4),\n        "released_undirected_edges": len(re_), "our_undirected_edges": len(oe),\n        "edge_recall": round(len(oe & re_) / max(1, len(re_)), 4),\n        "edge_recall_on_shared_nodes": round(len(oe & re_in) / max(1, len(re_in)), 4),\n        "edge_precision": round(len(oe & re_) / max(1, len(oe)), 4),\n        "released_edges_all_reciprocal": all((e["target"], e["source"]) in {(x["source"], x["target"]) for x in r["edges"]}\n                                             for e in r["edges"]),\n    }\n\n\nif __name__ == "__main__":\n    if sys.argv[1] == "build":\n        print(build(sys.argv[2], sys.argv[3], sys.argv[4], sys.argv[5], fixed="--fixed" in sys.argv))\n    else:\n        print(json.dumps(fidelity(sys.argv[2], sys.argv[3])))\n'); print('wrote common/graph_build.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/github_text.py', 'w').write('"""Fill problem_statement / created_at from GitHub (issue first, PR body as fallback).\n\nRun where api.github.com is reachable (Kaggle notebook with internet, or a laptop). Optional\nGITHUB_TOKEN env var raises the rate limit; unauthenticated (60 req/h) suffices for ~25 tasks.\n\nproblem_statement follows the SWE-bench convention: "<title>\\n<body>". created_at is the issue\'s\ncreation time (contamination dating). Provenance is recorded in meta: issue | pr_body.\n\nUsage: python common/github_text.py TASKS.jsonl META.jsonl OUT_TASKS.jsonl OUT_META.jsonl\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\n\n\ndef gh(path: str) -> dict | None:\n    req = urllib.request.Request(f"https://api.github.com{path}",\n                                 headers={"Accept": "application/vnd.github+json", "User-Agent": "heldout-builder"})\n    if os.environ.get("GITHUB_TOKEN"):\n        req.add_header("Authorization", f"Bearer {os.environ[\'GITHUB_TOKEN\']}")\n    for attempt in range(4):\n        try:\n            return json.loads(urllib.request.urlopen(req, timeout=30).read())\n        except urllib.error.HTTPError as e:\n            if e.code == 404:\n                return None\n            if e.code in (403, 429):\n                time.sleep(min(60, 2 ** attempt * 5))\n                continue\n            raise\n    return None\n\n\ndef main():\n    tasks = [json.loads(l) for l in open(sys.argv[1])]\n    meta = {m["instance_id"]: m for m in map(json.loads, open(sys.argv[2]))}\n    with open(sys.argv[3], "w") as ft, open(sys.argv[4], "w") as fm:\n        for t in tasks:\n            m = meta[t["instance_id"]]\n            text, src, created = None, None, None\n            for n in list(m.get("issue_refs", [])) + list(m.get("issue_refs_in_tests", [])):\n                it = gh(f"/repos/{t[\'repo\']}/issues/{n}")\n                if it and "pull_request" not in it:  # a real issue, not a PR\n                    text, src, created = f"{it[\'title\']}\\n{it.get(\'body\') or \'\'}", f"issue#{n}", it["created_at"]\n                    break\n            if text is None and m.get("pr_number"):\n                pr = gh(f"/repos/{t[\'repo\']}/pulls/{m[\'pr_number\']}")\n                if pr:\n                    text, src, created = f"{pr[\'title\']}\\n{pr.get(\'body\') or \'\'}", f"pr_body#{m[\'pr_number\']}", pr["created_at"]\n            if text is not None:\n                t["problem_statement"], t["created_at"] = text.replace("\\r\\n", "\\n"), created\n                m.update(needs_issue_text=False, problem_statement_source=src)\n            ft.write(json.dumps(t) + "\\n")\n            fm.write(json.dumps(m) + "\\n")\n            print(t["instance_id"], src, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote common/github_text.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/export_swebench.py', 'w').write('"""Export swegemma-format tasks to SWE-bench instance format (generality beyond this competition).\n\nDifferences handled:\n  * FAIL_TO_PASS / PASS_TO_PASS become JSON-encoded strings (SWE-bench convention);\n  * patches get standard `diff --git` headers back (swegemma dev style omits them; both apply with git);\n  * `version` / `environment_setup_commit` are filled from the base commit (one env per task);\n  * the swegemma-only signal (snapshot path, graph path) goes under `extra`.\n\nUsage: python -I common/export_swebench.py TASKS_WITH_F2P.jsonl OUT.jsonl\n"""\nimport json\nimport re\nimport sys\n\n\ndef add_git_headers(patch: str) -> str:\n    out = []\n    for line in patch.splitlines():\n        m = re.match(r"^--- a/(.+)$", line)\n        if m:\n            out.append(f"diff --git a/{m.group(1)} b/{m.group(1)}")\n        out.append(line)\n    return "\\n".join(out) + "\\n"\n\n\ndef main():\n    with open(sys.argv[2], "w") as out:\n        for line in open(sys.argv[1]):\n            t = json.loads(line)\n            out.write(json.dumps({\n                "instance_id": t["instance_id"].replace("_", "__", 1),\n                "repo": t["repo"], "base_commit": t["base_commit"],\n                "patch": add_git_headers(t["patch"]), "test_patch": add_git_headers(t["test_patch"]),\n                "problem_statement": t["problem_statement"], "hints_text": t.get("hints_text", ""),\n                "created_at": t.get("created_at", ""),\n                "version": t["base_commit"][:7], "environment_setup_commit": t["base_commit"],\n                "FAIL_TO_PASS": json.dumps(list(t.get("FAIL_TO_PASS", []))),\n                "PASS_TO_PASS": json.dumps(list(t.get("PASS_TO_PASS", []))),\n                "extra": {"swegemma_instance_id": t["instance_id"],\n                          "graph_file": f"graphs/{t[\'repo\'].split(\'/\')[1].lower()}_{t[\'base_commit\']}.json"},\n            }) + "\\n")\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote common/export_swebench.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d', exist_ok=True)
open('/kaggle/working/pd/pilot_d/__init__.py', 'w').write(''); print('wrote pilot_d/__init__.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d', exist_ok=True)
open('/kaggle/working/pd/pilot_d/build_tasks.py', 'w').write('"""Pilot D: turn mined candidates into competition-format tasks (+ sidecar metadata).\n\nWrites, under OUT_DIR (outside git):\n  tasks.jsonl            competition fields (+ optional FAIL_TO_PASS/PASS_TO_PASS later)\n  meta.jsonl             provenance: fix commit, PR, issue refs, dates, AI marker, timings\n  snapshots/<iid>.tgz    dev-layout snapshot\nproblem_statement is left EMPTY with needs_issue_text=true: issue text must be fetched from the\nGitHub API (common/github_text.py), which this cloud session cannot reach for third-party repos.\n\nUsage: python -I pilot_d/build_tasks.py CANDIDATES.jsonl REPO_DIR OUT_DIR [--n N] [--issue-linked-only]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom common.snapshot import build_snapshot  # noqa: E402\n\n\ndef spread(rows, n):\n    """Pick n rows spread evenly over time (rows sorted by commit date)."""\n    rows = sorted(rows, key=lambda r: r["committed_at"])\n    if len(rows) <= n:\n        return rows\n    step = len(rows) / n\n    return [rows[int(i * step)] for i in range(n)]\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("candidates")\n    p.add_argument("repo_dir")\n    p.add_argument("out_dir")\n    p.add_argument("--n", type=int, default=12)\n    p.add_argument("--issue-linked-only", action="store_true")\n    p.add_argument("--ids", nargs="*", help="build exactly these instance_ids (e.g. from pilot_d/manifest.json)")\n    a = p.parse_args()\n    rows = [json.loads(l) for l in open(a.candidates)]\n    if a.ids:\n        chosen = [r for r in rows if r["instance_id"] in set(a.ids)]\n    else:\n        if a.issue_linked_only:\n            rows = [r for r in rows if r["issue_refs"]]\n        chosen = spread(rows, a.n)\n    out = Path(a.out_dir)\n    (out / "snapshots").mkdir(parents=True, exist_ok=True)\n    with open(out / "tasks.jsonl", "a") as ft, open(out / "meta.jsonl", "a") as fm:\n        for r in chosen:\n            t0 = time.perf_counter()\n            snap = build_snapshot(a.repo_dir, r["base_commit"], str(out / "snapshots" / f"{r[\'instance_id\']}.tgz"))\n            task = {\n                "instance_id": r["instance_id"], "repo": r["repo"], "base_commit": r["base_commit"],\n                "patch": r["patch"], "test_patch": r["test_patch"],\n                "problem_statement": "", "hints_text": "", "created_at": r["authored_at"],\n            }\n            meta = {k: r.get(k) for k in ("instance_id", "repo", "fix_commit", "pr_number", "issue_refs",\n                                      "issue_refs_in_tests", "subject",\n                                      "committed_at", "authored_at", "code_files", "test_files",\n                                      "code_lines_changed", "ai_authored_marker")}\n            meta.update(snap, needs_issue_text=True, build_seconds=round(time.perf_counter() - t0, 1))\n            ft.write(json.dumps(task) + "\\n")\n            fm.write(json.dumps(meta) + "\\n")\n            print(r["instance_id"], snap, meta["build_seconds"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote pilot_d/build_tasks.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d', exist_ok=True)
open('/kaggle/working/pd/pilot_d/manifest.json', 'w').write('{\n "since": "2026-05-01",\n "clone_shallow_since": "2026-03-15",\n "note": "Held-out pilot tasks: instance_id = <repo_short>_<PR number or fix-commit prefix>. Rebuilt from public GitHub by pilot_d/build_tasks.py --ids.",\n "repos": [\n  {\n   "short": "sqlglot",\n   "repo": "tobymao/sqlglot",\n   "url": "https://github.com/tobymao/sqlglot",\n   "pkg": "sqlglot",\n   "licence": "MIT",\n   "instance_ids": [\n    "sqlglot_17432c6117",\n    "sqlglot_7640",\n    "sqlglot_7809",\n    "sqlglot_169b8364b6",\n    "sqlglot_df5ec1ecd4",\n    "sqlglot_8138",\n    "sqlglot_40f285cb61",\n    "sqlglot_8267",\n    "sqlglot_8321",\n    "sqlglot_a3884168f6",\n    "sqlglot_8447",\n    "sqlglot_8538",\n    "sqlglot_8596",\n    "sqlglot_8686"\n   ],\n   "fix_commits": {\n    "sqlglot_17432c6117": "17432c61175194ee0f56f0703393e8c5fdc9e81f",\n    "sqlglot_7640": "63296846f6a7a683af21683acb24aad7a63bb3f5",\n    "sqlglot_7809": "cf7e4bf4a57d331e4a6e5f4933cb7daf0e503c3b",\n    "sqlglot_169b8364b6": "169b8364b643c4285dcf073aa2f357c5b40831a6",\n    "sqlglot_df5ec1ecd4": "df5ec1ecd43b83ac63800bf40b04362c9af0173e",\n    "sqlglot_8138": "1dae76b5833a47c9e2bc0a72a93d94d829377a0c",\n    "sqlglot_40f285cb61": "40f285cb61edff4a40950fc708e79ae4e8ed583d",\n    "sqlglot_8267": "3110e151bfe6e229ec6b85708a56a9340a56c6a6",\n    "sqlglot_8321": "e27b7654a9d710e29da6f42994de845a99444c84",\n    "sqlglot_a3884168f6": "a3884168f639633fa8f64334bf63e4b343bedfe2",\n    "sqlglot_8447": "b3b6c57cea516bf8f76c6262b308d86dff82ca1b",\n    "sqlglot_8538": "e1e0d2e52f45247bfa0ab815eb8a7388ea50a0e8",\n    "sqlglot_8596": "e38929edd78a20f74fc19fee819e14ac23add687",\n    "sqlglot_8686": "fdd0e9a464f0f9ec425c56ae83fc87a5e2052087"\n   }\n  },\n  {\n   "short": "networkx",\n   "repo": "networkx/networkx",\n   "url": "https://github.com/networkx/networkx",\n   "pkg": "networkx",\n   "licence": "BSD-3-Clause",\n   "instance_ids": [\n    "networkx_8659",\n    "networkx_8726",\n    "networkx_8750",\n    "networkx_8807",\n    "networkx_8813",\n    "networkx_8837",\n    "networkx_8879",\n    "networkx_8881",\n    "networkx_8883",\n    "networkx_8899",\n    "networkx_8890",\n    "networkx_8906"\n   ],\n   "fix_commits": {\n    "networkx_8659": "e187ba20b38583a329c960062e4b9e0f23c8fc74",\n    "networkx_8726": "e9eac8a7e04df0b9cdcff8ed1682b3971678f4ed",\n    "networkx_8750": "b1f956344071ba1adfad17d56d62a7a78dfd37e6",\n    "networkx_8807": "2ebe5f80a114a181e03e18acc9491e45346598c0",\n    "networkx_8813": "2aecdbdd18210c38dc2b9fe32da3825bf30a3c33",\n    "networkx_8837": "416c3e26da0567c9242aa415810dcffca27d625f",\n    "networkx_8879": "73cb92bbaed12828848f2609340442119e7e0fe8",\n    "networkx_8881": "c279073cc33dab38777e701370210d0a7a98849a",\n    "networkx_8883": "0db8227000872d7a9f6ce84c54ba1e5e99429122",\n    "networkx_8899": "31ab6fd7e1e149e887282b83a91752c486d071b6",\n    "networkx_8890": "c94928ed94899033126c9d47f797a1f698584b20",\n    "networkx_8906": "9f6079c0120416599ea5d4018bbaba229ea2e456"\n   }\n  }\n ]\n}'); print('wrote pilot_d/manifest.json')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d/env', exist_ok=True)
open('/kaggle/working/pd/pilot_d/env/sqlglot.txt', 'w').write('# Test-only dependencies of tobymao/sqlglot (from setup.py extras "dev", imported by tests/)\nduckdb==1.4.1\npandas==2.3.3\npytz==2025.2\npython-dateutil==2.9.0.post0\n'); print('wrote pilot_d/env/sqlglot.txt')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d/env', exist_ok=True)
open('/kaggle/working/pd/pilot_d/env/networkx.txt', 'w').write('# Test-only dependencies of networkx/networkx used via pytest.importorskip (requirements/test.txt + extras)\nnumpy==2.3.4\nscipy==1.16.3\npandas==2.3.3\n'); print('wrote pilot_d/env/networkx.txt')

## Part A: gold-patch validity of all dev tasks (real Kaggle host env)
Expected runtime about 30–60 min on 4 CPUs. Per task: empty patch must fail, gold patch must pass.

In [ ]:
os.chdir('/kaggle/working/pd')
t0 = time.time()
r = subprocess.run([sys.executable, 'common/validate.py', '--tasks', str(DATA_DIR/'tasks.jsonl'), '--snapshots', str(DATA_DIR/'snapshots'),
                    '--wheels', str(DATA_DIR/'wheels'), '--backend', 'subprocess', '--concurrency', '4',
                    '--out', str(W/'dev_audit_per_task_DO_NOT_COMMIT.jsonl')], capture_output=True, text=True)
print('minutes', round((time.time()-t0)/60, 1)); print(r.stderr[-2000:])
from collections import Counter, defaultdict
rows = [json.loads(l) for l in open(W/'dev_audit_per_task_DO_NOT_COMMIT.jsonl')]
agg = defaultdict(Counter)
for x in rows:
    k = 'valid' if x['valid'] else ('empty_patch_already_passes' if x.get('base_resolved') else ('setup_error' if x.get('gold_resolved') is None else 'gold_patch_fails'))
    agg[x['repo']][k] += 1
symptoms = Counter((x['repo'], x.get('gold_symptom') or x.get('gold_error', '')[:60]) for x in rows if not x['valid'])
summary = {'part_A_dev_audit': {r_: dict(c) for r_, c in agg.items()},
           'part_A_failure_symptoms': [[a, b, n] for (a, b), n in symptoms.most_common(20)],
           'python': sys.version.split()[0]}
json.dump(summary, open(W/'pilot_d_summary.json', 'w'), indent=1); print(json.dumps(summary, indent=1))

## Part B: rebuild and validate the held-out pilot tasks (Internet ON)

In [ ]:
os.chdir('/kaggle/working/pd')
man = json.load(open('pilot_d/manifest.json'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'git+https://github.com/danielmantiglia/codegraph-loc@1e8e0abbb97a1f78648cb12890fd700712b00623'], check=True)
# Install declared test deps ONLY if missing (pinning would downgrade the Kaggle image's numpy/pandas); record versions used
import importlib, importlib.metadata as md
used = {}
for env in ('pilot_d/env/sqlglot.txt', 'pilot_d/env/networkx.txt'):
    for line in open(env):
        line = line.split('#')[0].strip()
        if not line: continue
        dist = line.split('==')[0]; mod = {'python-dateutil': 'dateutil'}.get(dist, dist)
        try: importlib.import_module(mod)
        except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', line], check=True)
        used[dist] = md.version(dist)
print('test deps used:', used)
H = W/'heldout'; H.mkdir(exist_ok=True)
for repo in man['repos']:
    d = Path('/kaggle/working/src')/repo['short']
    if not d.exists():
        subprocess.run(['git', 'clone', '-q', f"--shallow-since={man['clone_shallow_since']}", repo['url'], str(d)], check=True)
    subprocess.run([sys.executable, 'common/mining.py', str(d), '--repo', repo['repo'], '--pkg', repo['pkg'], '--since', man['since'],
                    '--out', str(H/f"cand_{repo['short']}.jsonl"), '--funnel', str(H/f"funnel_{repo['short']}.json")], check=True)
    subprocess.run([sys.executable, 'pilot_d/build_tasks.py', str(H/f"cand_{repo['short']}.jsonl"), str(d), str(H), '--ids', *repo['instance_ids']], check=True)
subprocess.run([sys.executable, 'common/github_text.py', str(H/'tasks.jsonl'), str(H/'meta.jsonl'), str(H/'tasks_text.jsonl'), str(H/'meta_text.jsonl')], check=True)
for step in (['common/validate.py', '--tasks', str(H/'tasks_text.jsonl'), '--snapshots', str(H/'snapshots'), '--backend', 'subprocess', '--out', str(H/'val_kaggle.jsonl')],
             ['common/testmap.py', '--tasks', str(H/'tasks_text.jsonl'), '--snapshots', str(H/'snapshots'), '--out', str(H/'testmap.jsonl')]):
    subprocess.run([sys.executable, *step], check=True)
tm = {json.loads(l)['instance_id']: json.loads(l) for l in open(H/'testmap.jsonl')}
with open(H/'tasks_final.jsonl', 'w') as out:
    for l in open(H/'tasks_text.jsonl'):
        t = json.loads(l); t['FAIL_TO_PASS'] = tm[t['instance_id']]['FAIL_TO_PASS']; t['PASS_TO_PASS'] = tm[t['instance_id']]['PASS_TO_PASS']; out.write(json.dumps(t)+'\n')
subprocess.run([sys.executable, 'common/annotate.py', '--tasks', str(H/'tasks_final.jsonl'), '--snapshots', str(H/'snapshots'),
                '--testmap', str(H/'testmap.jsonl'), '--out', str(H/'annotations.jsonl'), '--verify'], check=True)
import tarfile, tempfile
for l in open(H/'tasks_final.jsonl'):
    t = json.loads(l); short = t['repo'].split('/')[1].lower()
    with tempfile.TemporaryDirectory() as td:
        tarfile.open(H/'snapshots'/f"{t['instance_id']}.tgz").extractall(td, filter='data')
        subprocess.run([sys.executable, 'common/graph_build.py', 'build', td, short, t['base_commit'], str(H/'graphs')], check=True)
val = [json.loads(l) for l in open(H/'val_kaggle.jsonl')]
ann = [json.loads(l) for l in open(H/'annotations.jsonl')]
meta = [json.loads(l) for l in open(H/'meta_text.jsonl')]
summary = json.load(open(W/'pilot_d_summary.json'))
summary['part_B_heldout'] = {'tasks': len(val), 'valid_kaggle_subprocess': sum(v['valid'] for v in val),
    'per_task': [{k: v.get(k) for k in ('instance_id', 'valid', 'base_resolved', 'gold_resolved', 'gold_symptom')} for v in val],
    'repro_verified': sum(1 for a in ann if a.get('repro_test', {}).get('verified')),
    'problem_statement_source': Counter((m.get('problem_statement_source') or 'pending').split('#')[0] for m in meta),
    'test_deps_used': used}
json.dump(summary, open(W/'pilot_d_summary.json', 'w'), indent=1); print(json.dumps(summary['part_B_heldout'], indent=1))